# Segment 4 — Inference & Hyperparameters (Hands-on Playground)

This notebook is a companion demo for the lesson segment on **inference** and **decoding hyperparameters**.

It uses **Ollama** locally (HTTP API) to run the same prompt under different decoding settings and observe:

- Deterministic vs. stochastic generation
- Temperature effects
- Top-K and Top-P (nucleus) sampling
- Output length controls (`num_predict`, `stop` sequences)
- Repetition / degeneration and how penalties help
- Why agentic chains amplify variance across repeated calls
- Simple cost/latency proxies (token count approximations and elapsed time)

## Prerequisites

1. Install Ollama and pull a model (example):
```bash
ollama pull llama3.1:latest
```
2. Ensure Ollama is running (default endpoint: `http://localhost:11434`).

Notes:
- Token counts below are approximate unless you add a real tokenizer.
- All demos are written to be readable and editable in class.


## 0) Setup

We’ll call Ollama’s `generate` endpoint via HTTP and wrap it in a helper function.


In [1]:
import json
import time
import textwrap
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:latest"  # change if needed


def call_ollama(
    prompt: str,
    model: str = MODEL,
    system: Optional[str] = None,
    options: Optional[Dict[str, Any]] = None,
    stream: bool = False,
) -> Tuple[str, float, Dict[str, Any]]:
    """Call Ollama /api/generate and return (text, elapsed_seconds, raw_json).

    Uses stream=False for simplicity and reproducibility in class.
    """
    payload: Dict[str, Any] = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
    }
    if system is not None:
        payload["system"] = system
    if options is not None:
        payload["options"] = options

    t0 = time.time()
    r = requests.post(OLLAMA_URL, json=payload, timeout=300)
    r.raise_for_status()
    data = r.json()
    dt = time.time() - t0
    return data.get("response", ""), dt, data


def pretty(text: str, width: int = 92) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


def approx_token_count(text: str) -> int:
    """Very rough token proxy: ~4 chars per token on English text is a common heuristic."""
    return max(1, int(len(text) / 4))


@dataclass
class RunResult:
    label: str
    text: str
    seconds: float
    approx_tokens: int
    options: Dict[str, Any]


def run_and_summarize(label: str, prompt: str, options: Dict[str, Any]) -> RunResult:
    out, dt, _raw = call_ollama(prompt=prompt, options=options)
    return RunResult(
        label=label,
        text=out.strip(),
        seconds=dt,
        approx_tokens=approx_token_count(out),
        options=options,
    )


print("Ready. If this cell errors, confirm Ollama is running at:", OLLAMA_URL)


Ready. If this cell errors, confirm Ollama is running at: http://localhost:11434/api/generate


### Sanity check: one call

In [2]:
prompt = "Explain quantum computing to a 10-year-old in 5 short sentences."
res = run_and_summarize(
    label="sanity",
    prompt=prompt,
    options={"temperature": 0.3, "top_p": 0.9, "num_predict": 120},
)
print(pretty(res.text))
print(f"elapsed={res.seconds:.2f}s, approx_tokens={res.approx_tokens}")


Here's an explanation of quantum computing that a 10-year-old can understand:

Quantum computers are super powerful machines that can solve problems that regular computers
can't. They use tiny particles called atoms or photons that can be in many places at the
same time, which is weird and cool! This means that quantum computers can try lots of
solutions to a problem all at once, making them way faster than regular computers. Imagine
you have a huge box full of different colored balls, and you need to find a specific ball -
a regular computer would look through the box one by one, but a quantum computer
elapsed=10.89s, approx_tokens=152


## 1) Deterministic vs. Stochastic Generation

**Goal:** Show how near-greedy decoding differs from sampling.

In practice:
- Lower `temperature` and lower `top_p` tends toward deterministic behavior.
- Higher values increase diversity and variance across runs.

We'll run the *same* prompt multiple times and compare.


In [3]:
prompt = "Give a one-paragraph explanation of quantum computing, then one analogy."
runs = []

# Near-deterministic
for i in range(3):
    runs.append(run_and_summarize(
        label=f"lowT_run{i+1}",
        prompt=prompt,
        options={"temperature": 0.1, "top_p": 0.8, "num_predict": 220},
    ))

# More stochastic
for i in range(3):
    runs.append(run_and_summarize(
        label=f"highT_run{i+1}",
        prompt=prompt,
        options={"temperature": 1.1, "top_p": 0.95, "num_predict": 220},
    ))

for r in runs:
    print("="*90)
    print(r.label, r.options, f"| {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(pretty(r.text))


lowT_run1 {'temperature': 0.1, 'top_p': 0.8, 'num_predict': 220} | 12.29s | approx_tokens=306
Here's an explanation and analogy for quantum computing:

Quantum computing is a new paradigm for processing information that uses the principles of
quantum mechanics to perform calculations and operations on data. Unlike classical
computers, which use bits (0s and 1s) to represent information, quantum computers use qubits
(quantum bits), which can exist in multiple states simultaneously. This allows quantum
computers to process vast amounts of information in parallel, making them potentially much
faster than classical computers for certain types of calculations. Quantum computing has the
potential to revolutionize fields such as cryptography, optimization, and simulation, but it
also requires new approaches to programming and error correction.

Here's an analogy: Imagine a library with an infinite number of books, each containing a
different possible solution to a complex problem. A classical

## 2) Temperature Effects (Qualitative + Similarity)

We’ll compare outputs under different temperatures and compute a **rough similarity** metric:
- Jaccard similarity over unique words (very approximate)
- Helps illustrate “outputs diverge more at higher temperature”


In [4]:
import re

def words(s: str) -> List[str]:
    return re.findall(r"[A-Za-z']+", s.lower())

def jaccard(a: str, b: str) -> float:
    A, B = set(words(a)), set(words(b))
    if not A and not B:
        return 1.0
    return len(A & B) / max(1, len(A | B))

prompt = "Explain what a transformer model is in simple terms, in 6 sentences."

temps = [0.2, 0.7, 1.2]
out_by_temp = {}

for t in temps:
    out_by_temp[t] = []
    for i in range(3):
        out_by_temp[t].append(run_and_summarize(
            label=f"T{t}_run{i+1}",
            prompt=prompt,
            options={"temperature": t, "top_p": 0.9, "num_predict": 220},
        ))

# Print one sample per temperature
for t in temps:
    r = out_by_temp[t][0]
    print("="*90)
    print(f"Temperature {t} sample | {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(pretty(r.text))

# Similarity within each temperature (run1 vs run2, run1 vs run3)
print("\nRough within-temp similarity (Jaccard over unique words):")
for t in temps:
    r1, r2, r3 = out_by_temp[t]
    s12 = jaccard(r1.text, r2.text)
    s13 = jaccard(r1.text, r3.text)
    print(f"T={t}: sim(run1,run2)={s12:.2f}, sim(run1,run3)={s13:.2f}")


Temperature 0.2 sample | 11.38s | approx_tokens=263
Here's an explanation of a transformer model in simple terms:

A transformer model is a type of artificial intelligence (AI) algorithm used for natural
language processing (NLP). It's called a "transformer" because it takes the input text and
transforms it into a new representation that can be understood by the machine. Unlike other
AI models, transformers don't rely on word or sentence order to understand meaning -
instead, they look at all the words in the text together to get a sense of the overall
context. This allows them to capture nuances like idioms, sarcasm, and figurative language
that might be tricky for other models to grasp. The transformer model uses self-attention
mechanisms to weigh the importance of each word in relation to every other word, which helps
it understand complex relationships between words. Overall, transformers are highly
effective at tasks like language translation, text summarization, and question answ

## 3) Top-K vs Top-P (Nucleus) Sampling

**Top-K:** keep only the K most likely tokens, then sample  
**Top-P:** keep the smallest set of tokens whose cumulative probability ≥ P

We’ll compare a few settings at the same temperature.


In [5]:
prompt = "Write a concise explanation of gradient descent, then give one everyday analogy."

configs = [
    ("Top-K small (k=20)", {"temperature": 0.9, "top_k": 20, "top_p": 1.0, "num_predict": 220}),
    ("Top-K larger (k=80)", {"temperature": 0.9, "top_k": 80, "top_p": 1.0, "num_predict": 220}),
    ("Top-P tighter (p=0.80)", {"temperature": 0.9, "top_p": 0.80, "top_k": 0, "num_predict": 220}),
    ("Top-P looser (p=0.95)", {"temperature": 0.9, "top_p": 0.95, "top_k": 0, "num_predict": 220}),
]

results = []
for label, opts in configs:
    results.append(run_and_summarize(label=label, prompt=prompt, options=opts))

for r in results:
    print("="*90)
    print(r.label, r.options, f"| {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(pretty(r.text))


Top-K small (k=20) {'temperature': 0.9, 'top_k': 20, 'top_p': 1.0, 'num_predict': 220} | 15.12s | approx_tokens=254
**Gradient Descent Explanation**

Gradient descent is an optimization algorithm used to minimize or maximize the value of a
function by iteratively adjusting a set of parameters. It works as follows:

1. Initialize the parameters (e.g., weights and bias) with random values
2. Compute the gradient of the objective function with respect to each parameter
3. Adjust each parameter in the opposite direction of the gradient, i.e., move it down the
steepness of the curve
4. Repeat steps 2-3 until convergence or a stopping criterion is reached

**Everyday Analogy**

Imagine trying to find the lowest point on a topographic map by walking downhill from
different starting points. Each step you take gives you new information about the slope of
the terrain (the gradient). You adjust your path at each step to avoid going up any hills,
effectively moving in the direction of steepest des

## 4) Interaction: Temperature × Top-P

We’ll demonstrate how settings can compound:
- Low T + low top-p → rigid
- High T + high top-p → chaotic
- High T + low top-p → conflicting (often unstable)


In [6]:
prompt = "Explain Wi-Fi roaming to a network engineer in one paragraph, then give a short checklist."

configs = [
    ("LowT + LowP (rigid)", {"temperature": 0.2, "top_p": 0.5, "num_predict": 240}),
    ("LowT + HighP (sweet spot)", {"temperature": 0.3, "top_p": 0.9, "num_predict": 240}),
    ("HighT + LowP (conflicting)", {"temperature": 1.2, "top_p": 0.5, "num_predict": 240}),
    ("HighT + HighP (chaotic)", {"temperature": 1.2, "top_p": 0.95, "num_predict": 240}),
]

for label, opts in configs:
    r = run_and_summarize(label=label, prompt=prompt, options=opts)
    print("="*90)
    print(label, opts, f"| {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(pretty(r.text))


LowT + LowP (rigid) {'temperature': 0.2, 'top_p': 0.5, 'num_predict': 240} | 14.42s | approx_tokens=316
Wi-Fi roaming allows mobile devices to seamlessly transition between different wireless
access points (APs) within a network as they move around, ensuring continuous connectivity
and minimizing disruptions. When a device roams, the client's association with the current
AP is broken, and it searches for a new AP to connect to. The process involves several
steps: the client detects nearby APs, selects the best one based on signal strength and
other criteria, and then re-associates with that AP. This transition should be transparent
to the user, with minimal or no disruption to their connectivity.

**Wi-Fi Roaming Checklist:**

1. **AP placement**: Ensure APs are strategically placed to provide adequate coverage and
overlap between cells.
2. **Channel planning**: Configure channels for each AP to minimize interference and
optimize signal strength.
3. **Roaming threshold settings**: Set 

## 5) Output Length Controls: `num_predict` and `stop`

We will show:
- Hard cap: `num_predict`
- Graceful stop: `stop` sequences


In [7]:
prompt = "Answer in JSON with keys: summary, risks, next_steps. Topic: 'deploying an LLM for customer support'."

r1 = run_and_summarize(
    label="No stop, larger cap",
    prompt=prompt,
    options={"temperature": 0.2, "top_p": 0.9, "num_predict": 260},
)

r2 = run_and_summarize(
    label="Stop sequence",
    prompt=prompt,
    options={"temperature": 0.2, "top_p": 0.9, "num_predict": 260, "stop": ["\n}\n", "\n}\r\n"]},
)

r3 = run_and_summarize(
    label="Over-constrained cap",
    prompt=prompt,
    options={"temperature": 0.2, "top_p": 0.9, "num_predict": 60},
)

for r in [r1, r2, r3]:
    print("="*90)
    print(r.label, r.options, f"| {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(r.text)


No stop, larger cap {'temperature': 0.2, 'top_p': 0.9, 'num_predict': 260} | 15.87s | approx_tokens=290
Here is the answer in JSON format:

```json
{
  "summary": {
    "title": "Deploying an LLM for Customer Support",
    "description": "Deploying a Large Language Model (LLM) for customer support can significantly improve response times, accuracy, and overall customer experience. However, it also introduces new risks and considerations that must be carefully managed."
  },
  "risks": [
    {
      "id": 1,
      "title": "Data Bias and Inaccuracy",
      "description": "LLMs are only as good as the data they're trained on. If the training data is biased or inaccurate, the model may perpetuate these issues, leading to poor customer support."
    },
    {
      "id": 2,
      "title": "Dependence on Technology",
      "description": "Over-reliance on LLMs can lead to decreased human empathy and understanding of complex customer issues. This may result in a negative impact on customer sa

## 6) Repetition & Degeneration + Penalties

We’ll provoke repetition and then apply `frequency_penalty` and `presence_penalty`.


In [8]:
prompt = (
    "Continue the sentence for a long time while staying on the same topic: "
    "'The result is that'. Write 180 tokens."
)

configs = [
    ("No penalties", {"temperature": 0.9, "top_p": 0.95, "num_predict": 180}),
    ("Frequency penalty", {"temperature": 0.9, "top_p": 0.95, "frequency_penalty": 0.5, "num_predict": 180}),
    ("Presence penalty", {"temperature": 0.9, "top_p": 0.95, "presence_penalty": 0.5, "num_predict": 180}),
    ("Both (start low)", {"temperature": 0.9, "top_p": 0.95, "frequency_penalty": 0.3, "presence_penalty": 0.3, "num_predict": 180}),
]

for label, opts in configs:
    r = run_and_summarize(label=label, prompt=prompt, options=opts)
    print("="*90)
    print(label, opts, f"| {r.seconds:.2f}s | approx_tokens={r.approx_tokens}")
    print(pretty(r.text))


No penalties {'temperature': 0.9, 'top_p': 0.95, 'num_predict': 180} | 11.37s | approx_tokens=263
The result is that the lack of diversity in these groups can lead to a narrow perspective,
which can hinder innovation and progress. This limited thinking can further exacerbate
existing social and economic inequalities, resulting in systemic injustices that are
difficult to reverse. As a consequence, marginalized communities may feel disempowered and
disenfranchised, leading to increased frustration and disillusionment with the system.

This disillusionment can manifest as decreased participation in civic activities, lower
voter turnout, and a sense of hopelessness among certain groups. In turn, this decrease in
civic engagement can contribute to a self-perpetuating cycle of disengagement, where
individuals feel increasingly disconnected from the democratic process and less invested in
creating positive change.

Furthermore, the lack of representation in decision-making positions can perp

## 7) Agentic Systems: Variance Compounds Across Calls

We simulate an “agent loop” with multiple inference calls.
Each call takes the prior output as context and produces the next step.


In [9]:
def agent_chain(user_request: str, steps: int, options: Dict[str, Any]):
    chain = []
    context = f"User request: {user_request}\n"
    for i in range(steps):
        prompt = (
            context
            + f"Step {i+1}: Provide the next action as a single bullet. "
              "If you need a tool, write: TOOL: <name>(<args>). Otherwise write: THINK: <short>."
        )
        r = run_and_summarize(label=f"step{i+1}", prompt=prompt, options=options)
        chain.append(r)
        context += f"Assistant step {i+1}: {r.text}\n"
    return chain

request = "Find the latest guidance on benchmarking LLMs, then summarize in 3 bullet points."

stable_opts = {"temperature": 0.1, "top_p": 0.8, "num_predict": 80, "stop": ["\n\n"]}
noisy_opts  = {"temperature": 0.9, "top_p": 0.95, "num_predict": 80, "stop": ["\n\n"]}

stable_chain = agent_chain(request, steps=6, options=stable_opts)
noisy_chain  = agent_chain(request, steps=6, options=noisy_opts)

print("STABLE CHAIN (low variance)")
for r in stable_chain:
    print("-", r.text)

print("\nNOISY CHAIN (higher variance)")
for r in noisy_chain:
    print("-", r.text)


STABLE CHAIN (low variance)
- Here's the next step:
- • TOOL: Google Scholar (search query: "latest benchmarking LLMs")
- Here's the updated response:
- Here are the next steps:
- Here is the updated response:
- Here's the latest guidance on benchmarking LLMs:

NOISY CHAIN (higher variance)
- Here are the steps to find the latest guidance on benchmarking LLMs and summarize it in 3 bullet points:
- • Search online for the latest academic papers, research studies, or conference proceedings on benchmarking Large Language Models (LLMs).
• Check websites of organizations such as the Association for the Advancement of Artificial Intelligence (AAAI) and the International Joint Conference on Artificial Intelligence (IJCAI) for recent articles or presentations related to LLM benchmarking.
- THINK: Summarize key findings in a concise manner for easy understanding and comparison.
- Here is the revised assistant response:
- • Search online academic papers, research studies, and conference proceedi

## 8) Cost / Latency / Throughput Proxies

We measure:
- Elapsed time
- Approx token count (rough)
and compute a crude “tokens per second” proxy.


In [10]:
import pandas as pd

prompt = "Explain the difference between top-k and top-p in 6 sentences."

configs = [
    ("short, lowT", {"temperature": 0.2, "top_p": 0.9, "num_predict": 120}),
    ("long, lowT",  {"temperature": 0.2, "top_p": 0.9, "num_predict": 360}),
    ("short, highT", {"temperature": 1.0, "top_p": 0.95, "num_predict": 120}),
    ("long, highT",  {"temperature": 1.0, "top_p": 0.95, "num_predict": 360}),
]

rows = []
for label, opts in configs:
    r = run_and_summarize(label=label, prompt=prompt, options=opts)
    tps = r.approx_tokens / max(1e-6, r.seconds)
    rows.append({
        "label": label,
        "temperature": opts.get("temperature"),
        "top_p": opts.get("top_p"),
        "num_predict": opts.get("num_predict"),
        "elapsed_s": round(r.seconds, 3),
        "approx_tokens": r.approx_tokens,
        "approx_tokens_per_s": round(tps, 1),
    })

df = pd.DataFrame(rows).sort_values(["elapsed_s", "approx_tokens"])
df


,label,temperature,top_p,num_predict,elapsed_s,approx_tokens,approx_tokens_per_s
0,"short, lowT",0.2,0.90,120,9.455,158,16.7
2,"short, highT",1.0,0.95,120,10.745,153,14.2
3,"long, highT",1.0,0.95,360,15.709,238,15.2
1,"long, lowT",0.2,0.90,360,19.085,309,16.2
